In [ ]:
from docx import Document
from pathlib import Path

paths = {
    'target': Path('/workspace/Memoire_NAOMIS_restructure_usages_integres (2).docx'),
    'revised': Path('/workspace/Memoire_NAOMIS_Vatencul_LiDAR_WebSIG_revise.docx'),
}
for name,path in paths.items():
    doc=Document(path)
    print(f'\n{name}: paragraphs={len(doc.paragraphs)}, tables={len(doc.tables)}, sections={len(doc.sections)}')
    for i,p in enumerate(doc.paragraphs):
        t=p.text.strip()
        if t.startswith(('3.4.', 'CHAPITRE 5', 'CHAPITRE 6')):
            print(i, p.style.name, repr(t[:140]))


In [ ]:
from docx import Document
for label,path in paths.items():
    doc=Document(path)
    print('\n###',label)
    for i,p in enumerate(doc.paragraphs):
        if i>250 and p.style.name in ['NAOMIS - Chapitre','NAOMIS - Section','NAOMIS - Sous-section']:
            print(f'{i:3d} | {p.style.name:23s} | {p.text.strip()}')


In [ ]:
from docx import Document
p=paths['target']; doc=Document(p)
for i,para in enumerate(doc.paragraphs[327:],327):
    if para.style.name in ['NAOMIS - Chapitre','NAOMIS - Section','NAOMIS - Sous-section']:
        print(f'{i:3d} | {para.style.name:22s} | {para.text.strip()}')


In [ ]:
doc=Document(paths['target'])
for a,b,label in [(393,457,'Méthodologie du cas'),(521,581,'Résultats du cas')]:
    print('\n##',label)
    for i in range(a,b):
        p=doc.paragraphs[i]
        t=p.text.strip()
        if t:
            print(f'[{i}] ({p.style.name}) {t[:1200]}')


In [ ]:
doc=Document(paths['target'])
for a,b in [(415,457),(548,581)]:
    print(f'\n--- {a}:{b} ---')
    for i in range(a,b):
        p=doc.paragraphs[i]
        if p.text.strip(): print(f'[{i}] {p.style.name}: {p.text.strip()}')


In [ ]:
from docx import Document

doc=Document(paths['target'])
for i in range(548,561):
    p=doc.paragraphs[i]
    print(f'[{i}] {p.style.name}: {p.text.strip()}')
print('\nTables du document:')
for ti,t in enumerate(doc.tables):
    text=' | '.join(c.text.strip().replace('\n',' / ') for c in t.rows[0].cells)
    ncols=len(t.rows[0].cells) if t.rows else 0
    print(ti, len(t.rows), ncols, text[:180])


In [ ]:
from pathlib import Path
import pandas as pd, hashlib, json, os
base=Path('/workspace/vatencul_lidar_websig_outputs')
# Verify every checksum listed in the manifest against the actual files.
man=pd.read_csv(base/'manifest_sha256.csv')
print(man.columns.tolist(), man.shape)
print(man.head())
# Infer path/hash columns robustly
path_col=next(c for c in man.columns if 'fichier' in c.lower() or 'path' in c.lower())
hash_col=next(c for c in man.columns if 'sha' in c.lower())
checks=[]
for _,r in man.iterrows():
    fp=base/str(r[path_col])
    if fp.exists() and fp.is_file():
        h=hashlib.sha256()
        with open(fp,'rb') as f:
            for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
        checks.append((str(r[path_col]), h.hexdigest()==str(r[hash_col]).lower()))
    else:
        checks.append((str(r[path_col]), False))
print('Checksums:', sum(ok for _,ok in checks),'/',len(checks),'valides')
print('Échecs:', [p for p,ok in checks if not ok][:10])
# Sizes relevant to the dissemination package
for rel in ['vectors/vatencul_objets_websig.gpkg','mapstore_geoserver/batiments_extrusion.geojson','mapstore_geoserver/layer_catalog.json','mapstore_geoserver/mapstore_config_template.json']:
    fp=base/rel; print(rel, round(fp.stat().st_size/1024/1024,3),'Mio')


In [ ]:
from docx import Document
from docx.shared import Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from pathlib import Path
import shutil, re

src=Path('/workspace/Memoire_NAOMIS_restructure_usages_integres (2).docx')
out=Path('/workspace/Memoire_NAOMIS_restructure_usages_integres_MAJ_Cas_usage_Resultats.docx')
base=Path('/workspace/vatencul_lidar_websig_outputs')
doc=Document(src)

# Utilities that preserve the document's custom styles and insert in document order.
def setp(idx, text):
    doc.paragraphs[idx].text=text

def insert_after(paragraph, text='', style='Normal'):
    new_p = OxmlElement('w:p')
    paragraph._p.addnext(new_p)
    from docx.text.paragraph import Paragraph
    p=Paragraph(new_p, paragraph._parent)
    if style: p.style=style
    if text: p.add_run(text)
    return p

def insert_picture_after(paragraph, image_path, width_cm, caption):
    p=insert_after(paragraph)
    p.alignment=WD_ALIGN_PARAGRAPH.CENTER
    p.add_run().add_picture(str(image_path), width=Cm(width_cm))
    cap=insert_after(p, caption, 'Caption')
    return cap

# Front-matter status note: factual and no longer claims that archive checks are pending.
setp(0, "VERSION RESTRUCTURÉE MISE À JOUR — La méthodologie et les résultats du cas d’usage du Vatencul ont été actualisés à partir du paquet contrôlé. Les analyses de sensibilité à 1 m et 2 m et le déploiement fonctionnel complet GeoServer/MapStore restent à réaliser.")

# Strengthen and document the use-case methodology.
setp(417, "Deux surfaces sont distinguées. La délimitation de référence utilise les continuités hydroconditionnées et l’exutoire documenté, afin de conserver les sept raccords fonctionnels correspondant aux interruptions souterraines du Vatencul. Un second terrain, strictement diagnostique, est obtenu par remplissage généralisé des dépressions du MNT LiDAR brut à 0,50 m. Le premier sert à fixer le bassin relié à l’exutoire; le second sert à tester ce que restitue la topographie de surface sans forçage des ouvrages. Ils ne constituent donc pas deux estimations concurrentes d’une même variable.")
setp(419, "Sur le terrain diagnostique, chaque cellule reçoit une direction d’écoulement vers son voisin de plus forte pente selon l’algorithme D8, puis une aire contributive calculée à partir de la surface élémentaire de 0,25 m². Les axes sont extraits au seuil de 5 000 m². Les bâtiments dont la géométrie se situe à 10 m ou moins d’un axe et les tronçons routiers intersectés sont comptés sans les interpréter comme inondés. La profondeur de remplissage est la différence entre terrain rempli et terrain brut. Les composantes connexes sont retenues comme dépressions candidates lorsque leur profondeur atteint au moins 0,10 m et leur aire 10 m². Le volume intégré est uniquement géométrique.")
setp(421, "Les stations Météo-France à pas de six minutes sont classées selon leur distance au centroïde du bassin. La station la plus proche disposant d’une série exploitable est retenue. Les maxima glissants de 6 min, 1 h, 3 h, 6 h et 24 h sont calculés uniquement sur des fenêtres complètes. La fenêtre du maximum 24 h est exportée comme hyétogramme historique, sans période de retour ni transformation pluie-débit. Le paquet HEC-RAS rassemble terrain et géométries préparatoires, mais aucune simulation n’est lancée faute de paramètres hydrauliques et d’observations de calage.")
# Add explicit QC and reproducibility paragraph.
p421=doc.paragraphs[421]
insert_after(p421, "Tous les calculs sont réalisés en RGF93 / Lambert-93 (EPSG:2154); les altitudes sont conservées dans le référentiel IGN69. Les contrôles portent sur la validité du polygone, la présence de l’exutoire, le nombre de cellules, la cohérence surface-cellules, l’ouverture des rasters et GeoPackages, leur système de coordonnées et les empreintes SHA-256. Les seuils de 5 000 m², 10 m, 0,10 m et 10 m² sont des paramètres de présélection. Leur choix conditionne les objets détectés et n’a pas fait l’objet d’un étalonnage hydraulique.")

# Correct duplicated subsection numbering introduced by restructuring.
renames={
426:'3.9.1. Comparaison LAS–LAZ',429:'3.9.2. Comparaison contrôlée de deux implémentations',432:'3.9.3. Suivi des volumes et ressources dans la chaîne',
439:'3.10.1. Un choix de visualiseur fixé en amont',445:'3.10.2. Restitutions 2D et 3D',449:'3.10.3. Métadonnées et vérification fonctionnelle',453:'3.10.4. Reproductibilité et statut du démonstrateur',
470:'4.1.3. Couverture, densité et classification du nuage de points',471:'4.1.3.1. Une couverture presque continue, mais une densité de sol inégale',478:'4.1.3.2. Une classification dominée par le sol et la végétation',483:'4.1.3.3. Une concordance spatiale variable selon les objets',488:'4.1.3.4. Des absences de sol principalement associées au bâti',494:'4.1.4. Cohérence altimétrique des modèles numériques',495:'4.1.4.1. Une cohérence interne exacte entre MNT, MNS et MNH',498:'4.1.4.2. Des écarts généralement faibles entre le MNT et les points sol',503:'4.1.4.3. Un contrôle externe trop restreint pour conclure à l’exactitude absolue',507:'4.1.5. Synthèse du profil',
610:'5.8.1. Transférabilité aux usages secondaires',612:'5.8.1.1. Zones humides et plaines alluviales',617:'5.8.1.2. Désimperméabilisation',622:'5.8.2. Limites et portée des conclusions'}
for idx,text in renames.items(): setp(idx,text)

# Verified results, with clear distinction between observed output and interpretation.
setp(550, "Le polygone de référence est valide et couvre 730 611,75 m², soit 73,061 ha. L’exutoire retenu, X = 643 225,25 m et Y = 6 844 872,75 m en EPSG:2154, se trouve à 0,25 m de la limite du bassin et à 3,54 m du raccord avec l’Yvette. La surface correspond aux 2 922 447 cellules contributives de 0,25 m² enregistrées. Sur le MNT brut, l’altitude varie de 49,91 à 165,61 m (médiane 145,43 m; moyenne 128,83 m). La pente varie de 0 à 83,46° (médiane 5,93°; moyenne 10,28°); les valeurs extrêmes locales exigent un contrôle avant toute interprétation hydraulique.")
setp(552, "Au seuil de 5 000 m², le diagnostic D8 brut produit 11,411 km d’axes. Parmi les 398 bâtiments du bassin, 119, soit 29,9 %, se trouvent à 10 m ou moins d’un axe. Parmi 183 tronçons routiers, 66, soit 36,1 %, intersectent un axe. Quatre ponts sont recensés dans la BD TOPO. Ces fréquences décrivent des relations spatiales utiles à la priorisation des inspections; elles ne mesurent ni la probabilité d’inondation, ni la vulnérabilité, ni le risque.")
setp(554, "Le remplissage relève 350 123 des 2 922 447 cellules, soit 11,98 %. Parmi les cellules relevées, la profondeur médiane est de 0,079 m, le 95e percentile de 2,974 m et le maximum de 6,386 m. Le volume géométrique intégré atteint 42 314 m³. Après filtrage à 0,10 m de profondeur et 10 m² d’aire minimale, 188 dépressions candidates sont conservées. Ces valeurs ne représentent pas un stockage hydraulique mobilisable, car l’infiltration, les avaloirs, les conduites et les connexions souterraines ne sont pas modélisés.")
setp(556, "Au point exutoire documenté, le calcul sur le terrain brut rempli restitue une aire contributive presque nulle, contre 73,061 ha dans la couche hydroconditionnée de référence. Cet échec du diagnostic brut montre que le relief de surface et un remplissage généralisé ne rétablissent pas les tronçons souterrains. Il justifie le maintien explicite des sept raccords fonctionnels, à remplacer par la géométrie et les caractéristiques réelles des ouvrages dès qu’elles seront disponibles; il ne valide pas hydrauliquement la délimitation hydroconditionnée.")
setp(558, "La station de Gometz-le-Châtel (91275001), située à 6,58 km du centroïde, fournit 727 580 pas entre le 3 mai 2018 et le 21 août 2026, dont 99,19 % avec une valeur de précipitation. Sur les seules fenêtres complètes, les maxima glissants sont de 12,3 mm en 6 min, 35,8 mm en 1 h, 41,6 mm en 3 h, 44,2 mm en 6 h et 75,2 mm en 24 h. La fenêtre du 9 octobre 2024 à 05:36 UTC au 10 octobre à 05:30 UTC cumule 75,2 mm sans lacune. Elle constitue un événement historique de démonstration, non une pluie de projet et non une estimation de période de retour.")
setp(559, "Le paquet préparatoire HEC-RAS contient le MNT LiDAR, le bassin, l’exutoire, les tronçons visibles, les raccords souterrains documentés, le MOS 2025, les bâtiments, les routes, les ponts et l’hyétogramme historique. Aucune simulation défendable n’est produite. Il manque notamment le réseau pluvial vectoriel, les regards et avaloirs, les diamètres, radiers, pentes et états des conduites, la géométrie des buses, les conditions aval sur l’Yvette, des paramètres locaux d’infiltration et de rugosité et des observations événementielles indépendantes.")
setp(560, "Les valeurs ci-dessus ont été contrôlées dans les tableaux, les rasters et les GeoPackages du paquet livré. Les 38 empreintes du manifeste SHA-256 concordent avec les fichiers présents. Ce contrôle établit l’intégrité de la livraison analysée, pas la validité hydraulique des résultats.")

# Add land-use and imperviousness result after relief, before axes.
p550=doc.paragraphs[550]
h=insert_after(p550, '4.4.2. Occupation du sol et imperméabilisation', 'NAOMIS - Sous-section')
p=insert_after(h, "Le MOS 2025 décrit un bassin mixte. Les bois ou forêts couvrent 20,91 ha (28,62 %), les surfaces engazonnées entretenues 11,33 ha (15,51 %), les zones d’activités économiques 9,16 ha (12,54 %), l’habitat individuel 8,65 ha (11,84 %) et les voies routières 7,99 ha (10,94 %). Dans le raster Copernicus 2024, les 7 320 pixels valides ont une imperméabilisation moyenne de 29,91 %, une médiane de 0 % et 13,22 % atteignent au moins 80 %. Ces données contextualisent la production potentielle de ruissellement mais ne fournissent directement ni infiltration, ni rugosité, ni raccordement au réseau pluvial.")
# Shift following displayed subsection labels.
setp(551,'4.4.3. Axes topographiques et objets proches')
setp(553,'4.4.4. Dépressions candidates')
setp(555,'4.4.5. Continuité vers l’exutoire')
setp(557,'4.4.6. Contexte pluviométrique et préparation HEC-RAS')

# Honest status for absent analyses and corrected publication inventory.
setp(562, "Les résultats disponibles portent sur la grille de 0,50 m et la nouvelle emprise de 73,061 ha. Aucun raster ni tableau recalculé à 1 m et 2 m n’est présent dans le paquet contrôlé; les anciennes valeurs obtenues sur 71,86975 ha ne sont donc pas réutilisées. La sensibilité à la résolution reste une analyse planifiée et ne peut pas être conclue avec cette livraison.")
setp(564, "Le paquet de diffusion comprend cinq Cloud Optimized GeoTIFF (MNT, pente, ombrage, aire contributive et profondeur de remplissage), un GeoPackage de 20 couches, un catalogue JSON de sept entrées, une configuration modèle GeoServer/MapStore et un GeoJSON de bâtiments extrudables nécessitant encore une conversion en 3D Tiles. Les sources analytiques restent séparées des dérivés web.")
setp(565, "Le contrôle d’intégrité recense 38 fichiers et toutes les empreintes SHA-256 concordent. Le GeoPackage des objets WebSIG occupe 3,73 Mio et le GeoJSON préparatoire des bâtiments 0,71 Mio. Ces tailles décrivent la livraison présente; aucun temps d’accès réseau ou de rendu client n’a été mesuré.")
setp(568, "Le paquet de publication effectivement préparé contient cinq rasters Cloud Optimized GeoTIFF, un GeoPackage de 20 couches, un catalogue de couches et une configuration MapStore modèle. Les rasters proposés à GeoServer sont le MNT, la pente, l’ombrage, l’aire contributive D8 et la profondeur de remplissage. Les transformations destinées au Web ne remplacent pas les fichiers utilisés dans les calculs.")
setp(569, "Les couches vectorielles couvrent le bassin et l’exutoire, le Vatencul visible et ses raccords, le chemin aval, les axes, les dépressions, le bâti, les routes, les ponts, l’hydrographie, la végétation, le MOS, le PLU et le PPRI. Le catalogue prévoit une diffusion raster en WMS/WMTS et vectorielle en WFS ou OGC API Features. Cette organisation est documentée mais n’équivaut pas à un service déployé.")
setp(571, "La branche 2D est préparée pour GeoServer puis MapStore: les vecteurs peuvent être publiés en WMS/WFS ou OGC API Features, et les rasters en WMS/WMTS. Les adresses de services restent des variables à remplacer dans la configuration modèle. Aucune instance déployée n’est fournie dans la livraison.")
setp(572, "La branche 3D fournit un GeoJSON de bâtiments extrudables. Sa conversion en 3D Tiles, l’hébergement de ces tuiles et leur raccordement à MapStore restent nécessaires. Le terrain Cesium et une chaîne complète de visualisation 3D ne sont pas livrés ni testés; aucune performance de chargement ou amélioration de compréhension ne peut donc être affirmée.")
setp(576, "La livraison établit la disponibilité et l’intégrité d’un paquet cohérent de préparation à la publication 2D et 3D. Elle ne démontre pas le fonctionnement complet du démonstrateur, car l’instance GeoServer/MapStore, le terrain Cesium et les 3D Tiles ne sont pas fournis comme une chaîne déployée et testée.")
setp(577, "La validation fonctionnelle reste à conduire sur l’ouverture des couches, l’interrogation des attributs, la conservation des emprises, l’accès aux métadonnées, le passage 2D/3D et le chargement des actifs tuilés. En l’absence de test utilisateur comparatif, aucun gain de compréhension ou de décision ne peut être attribué à la 3D.")
setp(578, "Travaux restant à conduire : déployer GeoServer et MapStore; renseigner les URL de services; convertir les bâtiments en 3D Tiles; construire le terrain 3D; puis mesurer la disponibilité, l’interrogation, la fidélité géométrique et les temps de chargement selon un protocole reproductible.")
setp(580, "Le corpus général fournit une base techniquement exploitable, sous réserve du contrôle altimétrique externe limité. Sur 73,061 ha, le cas du Vatencul produit 11,411 km d’axes D8 bruts, présélectionne 119 bâtiments et 66 tronçons routiers, et identifie 188 dépressions candidates. L’échec du terrain brut à restituer l’aire contributive à l’exutoire souligne le rôle déterminant des continuités souterraines. Les produits WebSIG et HEC-RAS sont préparatoires; la sensibilité à 1 m et 2 m, la simulation hydraulique et le déploiement MapStore ne sont pas démontrés.")

# Correct one grammar issue.
# Update summary/abstract front matter to remove ambiguous pending archive verification while retaining limitations.
for p in doc.paragraphs:
    if 'la vérification fonctionnelle complète de MapStore doivent encore être finalisées' in p.text:
        p.text=p.text.replace('la vérification fonctionnelle complète de MapStore doivent encore être finalisées','la vérification fonctionnelle complète de MapStore restent à finaliser')
    if 'the complete functional verification of MapStore must still be finalised' in p.text:
        p.text=p.text.replace('the complete functional verification of MapStore must still be finalised','the complete functional verification of MapStore remain to be completed')

# Insert three verified figures at targeted updated result paragraphs. Avoid the flawed hypsometry plot.
insert_picture_after(doc.paragraphs[552], base/'figures/carte_diagnostic_vatencul.png', 15.5, 'Figure 4.X - Relief, axes D8 bruts, dépressions candidates et continuités documentées dans le bassin du Vatencul.')
insert_picture_after(p, base/'figures/occupation_sol_mos2025.png', 15.0, 'Figure 4.X - Dix principales occupations du sol dans le bassin du Vatencul, MOS 2025.')
insert_picture_after(doc.paragraphs[558], base/'figures/hyetogramme_gometz_2024_10_09.png', 15.5, 'Figure 4.X - Hyétogramme historique observé à Gometz-le-Châtel du 9 au 10 octobre 2024, utilisé comme donnée préparatoire.')

# Tell Word to refresh fields (TOC, lists, cross references) upon opening.
settings=doc.settings._element
update=OxmlElement('w:updateFields'); update.set('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val','true'); settings.append(update)

doc.save(out)
print(out)
print('size MiB',round(out.stat().st_size/1024/1024,2),'paragraphs',len(Document(out).paragraphs),'tables',len(Document(out).tables))


In [ ]:
from docx import Document
from pathlib import Path
from docx.oxml import OxmlElement

out=Path('/workspace/Memoire_NAOMIS_restructure_usages_integres_MAJ_Cas_usage_Resultats.docx')
doc=Document(out)

def delete_paragraph(p):
    el=p._element
    el.getparent().remove(el)
    p._p=p._element=None

def semantic_title(text):
    import re
    return re.sub(r'^\d+(?:\.\d+)*\.\s*', '', text.strip())

# Remove the superseded red-work-version banner retained as a floating/body duplicate.
for p in list(doc.paragraphs):
    if p.text.strip().startswith('VERSION DE TRAVAIL RESTRUCTURÉE —'):
        delete_paragraph(p)

# Remove adjacent duplicated headings caused by old numbering surviving restructuring.
paras=list(doc.paragraphs)
prev_heading=None
for p in list(paras):
    if p._element is None: continue
    if p.style.name in ('NAOMIS - Chapitre','NAOMIS - Section','NAOMIS - Sous-section') and p.text.strip():
        if prev_heading is not None and semantic_title(prev_heading.text)==semantic_title(p.text):
            delete_paragraph(p)
            continue
        prev_heading=p
    elif p.text.strip():
        prev_heading=None

# Replace the last remaining placeholder wording with a factual pending-analysis statement.
for p in doc.paragraphs:
    if p.text.strip().startswith('À COMPLÉTER / VÉRIFIER — recalculer sur les 73,061 ha'):
        p.text=('Analyse non disponible dans la livraison contrôlée — les indicateurs à 1 m et 2 m '
                '(écarts altimétriques, pentes, axes, croisements, tailles, durées et mémoire) devront être '
                'recalculés sur l’emprise de 73,061 ha avant toute conclusion sur la sensibilité à la résolution.')

# Normalize only the chapter-4 use-case subsection headings by semantic title.
case_map={
'Bassin, exutoire et relief':'4.4.1. Bassin, exutoire et relief',
'Occupation du sol et imperméabilisation':'4.4.2. Occupation du sol et imperméabilisation',
'Axes topographiques et objets proches':'4.4.3. Axes topographiques et objets proches',
'Dépressions candidates':'4.4.4. Dépressions candidates',
'Continuité vers l’exutoire':'4.4.5. Continuité vers l’exutoire',
'Contexte pluviométrique et préparation HEC-RAS':'4.4.6. Contexte pluviométrique et préparation HEC-RAS'}
in_case=False
for p in doc.paragraphs:
    t=p.text.strip()
    if t.startswith('4.4. Diagnostic topographique'): in_case=True
    elif in_case and t.startswith('4.5.'): in_case=False
    if in_case and semantic_title(t) in case_map:
        p.text=case_map[semantic_title(t)]

# Ensure fields refresh in Word.
settings=doc.settings._element
if not settings.xpath('./w:updateFields'):
    update=OxmlElement('w:updateFields')
    update.set('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val','true')
    settings.append(update)

doc.save(out)

# Verification report.
check=Document(out)
texts=[p.text.strip() for p in check.paragraphs if p.text.strip()]
print('paragraphs',len(check.paragraphs),'tables',len(check.tables),'size MiB',round(out.stat().st_size/1024/1024,2))
print('old banner',sum(t.startswith('VERSION DE TRAVAIL') for t in texts))
print('placeholder markers',sum('À COMPLÉTER / VÉRIFIER' in t for t in texts))
# Detect adjacent semantic duplicate headings.
heads=[(i,p.text.strip()) for i,p in enumerate(check.paragraphs) if p.style.name in ('NAOMIS - Chapitre','NAOMIS - Section','NAOMIS - Sous-section') and p.text.strip()]
dups=[]
for (i,a),(j,b) in zip(heads,heads[1:]):
    if j==i+1 and semantic_title(a)==semantic_title(b): dups.append((i,a,j,b))
print('adjacent duplicate headings',dups)
for i,t in heads:
    if t.startswith(('3.7.','3.8.','3.9.','3.10.','4.4.','4.5.')): print(i,t)


In [ ]:
from docx import Document
from pathlib import Path
import re
out=Path('/workspace/Memoire_NAOMIS_restructure_usages_integres_MAJ_Cas_usage_Resultats.docx')
doc=Document(out)
for i,p in enumerate(doc.paragraphs):
    t=p.text.strip()
    if (400 <= i <= 470 or 535 <= i <= 585) and t:
        print(f'{i:3d} | {p.style.name:22s} | {t[:180]}')
print('\nMarqueurs restants:')
for i,p in enumerate(doc.paragraphs):
    if 'À COMPLÉTER / VÉRIFIER' in p.text:
        print(i,p.text[:240])


In [ ]:
from docx import Document
from docx.shared import Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.text.paragraph import Paragraph
from pathlib import Path
import re

src=Path('/workspace/Memoire_NAOMIS_restructure_usages_integres (2).docx')
out=Path('/workspace/Memoire_NAOMIS_restructure_usages_integres_MAJ_Cas_usage_Resultats.docx')
base=Path('/workspace/vatencul_lidar_websig_outputs')
doc=Document(src)

# Helpers based on text identity, avoiding unstable paragraph indices.
def find_one(prefix):
    hits=[p for p in doc.paragraphs if p.text.strip().startswith(prefix)]
    if len(hits)!=1:
        raise ValueError(f'{prefix!r}: {len(hits)} occurrences')
    return hits[0]

def replace_prefix(prefix, new):
    p=find_one(prefix); p.text=new; return p

def insert_after(paragraph, text='', style='Normal'):
    el=OxmlElement('w:p'); paragraph._p.addnext(el)
    p=Paragraph(el, paragraph._parent); p.style=style
    if text: p.add_run(text)
    return p

def add_figure_after(paragraph, path, caption, width=15.5):
    p=insert_after(paragraph); p.alignment=WD_ALIGN_PARAGRAPH.CENTER
    p.add_run().add_picture(str(path), width=Cm(width))
    return insert_after(p, caption, 'Caption')

# Replace the working banner rather than adding a second one.
replace_prefix('VERSION DE TRAVAIL RESTRUCTURÉE',
"VERSION RESTRUCTURÉE MISE À JOUR — La méthodologie et les résultats du cas d’usage du Vatencul ont été actualisés à partir du paquet contrôlé. Les analyses de sensibilité à 1 m et 2 m et le déploiement fonctionnel complet GeoServer/MapStore restent à réaliser.")

# Methodology of the use case.
replace_prefix('La surface hydroconditionnée sert à délimiter',
"Deux surfaces sont distinguées. La délimitation de référence utilise les continuités hydroconditionnées et l’exutoire documenté, afin de conserver les sept raccords fonctionnels correspondant aux interruptions souterraines du Vatencul. Un second terrain, strictement diagnostique, est obtenu par remplissage généralisé des dépressions du MNT LiDAR brut à 0,50 m. Le premier sert à fixer le bassin relié à l’exutoire; le second sert à tester ce que restitue la topographie de surface sans forçage des ouvrages. Ils ne constituent donc pas deux estimations concurrentes d’une même variable.")
replace_prefix('Les directions D8 et l’aire contributive',
"Sur le terrain diagnostique, chaque cellule reçoit une direction d’écoulement vers son voisin de plus forte pente selon l’algorithme D8, puis une aire contributive calculée à partir de la surface élémentaire de 0,25 m². Les axes sont extraits au seuil de 5 000 m². Les bâtiments dont la géométrie se situe à 10 m ou moins d’un axe et les tronçons routiers intersectés sont comptés sans les interpréter comme inondés. La profondeur de remplissage est la différence entre terrain rempli et terrain brut. Les composantes connexes sont retenues comme dépressions candidates lorsque leur profondeur atteint au moins 0,10 m et leur aire 10 m². Le volume intégré est uniquement géométrique.")
p_rain_method=replace_prefix('La station Météo-France à six minutes la plus proche',
"Les stations Météo-France à pas de six minutes sont classées selon leur distance au centroïde du bassin. La station la plus proche disposant d’une série exploitable est retenue. Les maxima glissants de 6 min, 1 h, 3 h, 6 h et 24 h sont calculés uniquement sur des fenêtres complètes. La fenêtre du maximum 24 h est exportée comme hyétogramme historique, sans période de retour ni transformation pluie-débit. Le paquet HEC-RAS rassemble terrain et géométries préparatoires, mais aucune simulation n’est lancée faute de paramètres hydrauliques et d’observations de calage.")
insert_after(p_rain_method,
"Tous les calculs sont réalisés en RGF93 / Lambert-93 (EPSG:2154); les altitudes sont conservées dans le référentiel IGN69. Les contrôles portent sur la validité du polygone, la présence de l’exutoire, le nombre de cellules, la cohérence surface-cellules, l’ouverture des rasters et GeoPackages, leur système de coordonnées et les empreintes SHA-256. Les seuils de 5 000 m², 10 m, 0,10 m et 10 m² sont des paramètres de présélection. Leur choix conditionne les objets détectés et n’a pas fait l’objet d’un étalonnage hydraulique.")

# Correct numbering after restructuring.
heading_replacements={
'3.6.3. Comparaison LAS–LAZ':'3.9.1. Comparaison LAS–LAZ',
'3.6.4. Comparaison contrôlée de deux implémentations':'3.9.2. Comparaison contrôlée de deux implémentations',
'3.6.5. Suivi des volumes et ressources dans la chaîne':'3.9.3. Suivi des volumes et ressources dans la chaîne',
'3.7.1. Un choix de visualiseur fixé en amont':'3.10.1. Un choix de visualiseur fixé en amont',
'3.7.2. Restitutions 2D et 3D':'3.10.2. Restitutions 2D et 3D',
'3.7.3. Métadonnées et vérification fonctionnelle':'3.10.3. Métadonnées et vérification fonctionnelle',
'3.7.4. Reproductibilité et statut du démonstrateur':'3.10.4. Reproductibilité et statut du démonstrateur',
'4.1.2. Couverture, densité et classification du nuage de points':'4.1.3. Couverture, densité et classification du nuage de points',
'4.2.1. Une couverture presque continue, mais une densité de sol inégale':'4.1.3.1. Une couverture presque continue, mais une densité de sol inégale',
'4.2.2. Une classification dominée par le sol et la végétation':'4.1.3.2. Une classification dominée par le sol et la végétation',
'4.2.3. Une concordance spatiale variable selon les objets':'4.1.3.3. Une concordance spatiale variable selon les objets',
'4.2.4. Des absences de sol principalement associées au bâti':'4.1.3.4. Des absences de sol principalement associées au bâti',
'4.1.3. Cohérence altimétrique des modèles numériques':'4.1.4. Cohérence altimétrique des modèles numériques',
'4.3.1. Une cohérence interne exacte entre MNT, MNS et MNH':'4.1.4.1. Une cohérence interne exacte entre MNT, MNS et MNH',
'4.3.2. Des écarts généralement faibles entre le MNT et les points sol':'4.1.4.2. Des écarts généralement faibles entre le MNT et les points sol',
'4.3.3. Un contrôle externe trop restreint pour conclure à l’exactitude absolue':'4.1.4.3. Un contrôle externe trop restreint pour conclure à l’exactitude absolue',
'4.1.4. Synthèse du profil':'4.1.5. Synthèse du profil',
'6.5. Transférabilité aux usages secondaires':'5.8.1. Transférabilité aux usages secondaires',
'6.5.1. Zones humides et plaines alluviales':'5.8.1.1. Zones humides et plaines alluviales',
'6.5.2. Désimperméabilisation':'5.8.1.2. Désimperméabilisation',
'6.6. Limites et portée des conclusions':'5.8.2. Limites et portée des conclusions'}
for p in doc.paragraphs:
    if p.text.strip() in heading_replacements:
        p.text=heading_replacements[p.text.strip()]

# Verified case-use results.
p_relief=replace_prefix('Le bassin versant contrôlé couvre 73,061 ha.',
"Le polygone de référence est valide et couvre 730 611,75 m², soit 73,061 ha. L’exutoire retenu, X = 643 225,25 m et Y = 6 844 872,75 m en EPSG:2154, se trouve à 0,25 m de la limite du bassin et à 3,54 m du raccord avec l’Yvette. La surface correspond aux 2 922 447 cellules contributives de 0,25 m² enregistrées. Sur le MNT brut, l’altitude varie de 49,91 à 165,61 m (médiane 145,43 m; moyenne 128,83 m). La pente varie de 0 à 83,46° (médiane 5,93°; moyenne 10,28°); les valeurs extrêmes locales exigent un contrôle avant toute interprétation hydraulique.")
# Add a new subsection after the relief result.
h_land=insert_after(p_relief,'4.4.2. Occupation du sol et imperméabilisation','NAOMIS - Sous-section')
p_land=insert_after(h_land,
"Le MOS 2025 décrit un bassin mixte. Les bois ou forêts couvrent 20,91 ha (28,62 %), les surfaces engazonnées entretenues 11,33 ha (15,51 %), les zones d’activités économiques 9,16 ha (12,54 %), l’habitat individuel 8,65 ha (11,84 %) et les voies routières 7,99 ha (10,94 %). Dans le raster Copernicus 2024, les 7 320 pixels valides ont une imperméabilisation moyenne de 29,91 %, une médiane de 0 % et 13,22 % atteignent au moins 80 %. Ces données contextualisent la production potentielle de ruissellement mais ne fournissent directement ni infiltration, ni rugosité, ni raccordement au réseau pluvial.")

# Rename original case subsections by their exact titles.
for old,new in {
'4.4.2. Axes topographiques et objets proches':'4.4.3. Axes topographiques et objets proches',
'4.4.3. Dépressions candidates':'4.4.4. Dépressions candidates',
'4.4.4. Continuité vers l’exutoire':'4.4.5. Continuité vers l’exutoire',
'4.4.5. Contexte pluviométrique et préparation HEC-RAS':'4.4.6. Contexte pluviométrique et préparation HEC-RAS'}.items():
    find_one(old).text=new

p_axes=replace_prefix('Le diagnostic D8 sur le MNT non hydroconditionné',
"Au seuil de 5 000 m², le diagnostic D8 brut produit 11,411 km d’axes. Parmi les 398 bâtiments du bassin, 119, soit 29,9 %, se trouvent à 10 m ou moins d’un axe. Parmi 183 tronçons routiers, 66, soit 36,1 %, intersectent un axe. Quatre ponts sont recensés dans la BD TOPO. Ces fréquences décrivent des relations spatiales utiles à la priorisation des inspections; elles ne mesurent ni la probabilité d’inondation, ni la vulnérabilité, ni le risque.")
replace_prefix('Le filtrage à 0,10 m de profondeur',
"Le remplissage relève 350 123 des 2 922 447 cellules, soit 11,98 %. Parmi les cellules relevées, la profondeur médiane est de 0,079 m, le 95e percentile de 2,974 m et le maximum de 6,386 m. Le volume géométrique intégré atteint 42 314 m³. Après filtrage à 0,10 m de profondeur et 10 m² d’aire minimale, 188 dépressions candidates sont conservées. Ces valeurs ne représentent pas un stockage hydraulique mobilisable, car l’infiltration, les avaloirs, les conduites et les connexions souterraines ne sont pas modélisés.")
replace_prefix('Le calcul sur le terrain non hydroconditionné ne restitue',
"Au point exutoire documenté, le calcul sur le terrain brut rempli restitue une aire contributive presque nulle, contre 73,061 ha dans la couche hydroconditionnée de référence. Cet échec du diagnostic brut montre que le relief de surface et un remplissage généralisé ne rétablissent pas les tronçons souterrains. Il justifie le maintien explicite des sept raccords fonctionnels, à remplacer par la géométrie et les caractéristiques réelles des ouvrages dès qu’elles seront disponibles; il ne valide pas hydrauliquement la délimitation hydroconditionnée.")
p_rain_result=replace_prefix('La station de Gometz-le-Châtel, située à 6,58 km',
"La station de Gometz-le-Châtel (91275001), située à 6,58 km du centroïde, fournit 727 580 pas entre le 3 mai 2018 et le 21 août 2026, dont 99,19 % avec une valeur de précipitation. Sur les seules fenêtres complètes, les maxima glissants sont de 12,3 mm en 6 min, 35,8 mm en 1 h, 41,6 mm en 3 h, 44,2 mm en 6 h et 75,2 mm en 24 h. La fenêtre du 9 octobre 2024 à 05:36 UTC au 10 octobre à 05:30 UTC cumule 75,2 mm sans lacune. Elle constitue un événement historique de démonstration, non une pluie de projet et non une estimation de période de retour.")
replace_prefix('Aucune simulation HEC-RAS défendable n’est produite.',
"Le paquet préparatoire HEC-RAS contient le MNT LiDAR, le bassin, l’exutoire, les tronçons visibles, les raccords souterrains documentés, le MOS 2025, les bâtiments, les routes, les ponts et l’hyétogramme historique. Aucune simulation défendable n’est produite. Il manque notamment le réseau pluvial vectoriel, les regards et avaloirs, les diamètres, radiers, pentes et états des conduites, la géométrie des buses, les conditions aval sur l’Yvette, des paramètres locaux d’infiltration et de rugosité et des observations événementielles indépendantes.")

# Replace each unresolved marker with an explicit, scientifically honest status.
marker_updates={
'À COMPLÉTER / VÉRIFIER — vérifier ces résultats dans l’archive':
"Les valeurs ci-dessus ont été contrôlées dans les tableaux, les rasters et les GeoPackages du paquet livré. Les 38 empreintes du manifeste SHA-256 concordent avec les fichiers présents. Ce contrôle établit l’intégrité de la livraison analysée, pas la validité hydraulique des résultats.",
'À COMPLÉTER / VÉRIFIER — insérer les résultats recalculés à 1 m et 2 m':
"Les résultats disponibles portent sur la grille de 0,50 m et l’emprise de 73,061 ha. Aucun raster ni tableau recalculé à 1 m et 2 m n’est présent dans le paquet contrôlé; les anciennes valeurs obtenues sur 71,86975 ha ne sont donc pas réutilisées. La sensibilité à la résolution reste une analyse planifiée et ne peut pas être conclue avec cette livraison.",
'À COMPLÉTER / VÉRIFIER — compléter avec les tailles':
"Le contrôle d’intégrité recense 38 fichiers et toutes les empreintes SHA-256 concordent. Le GeoPackage des objets WebSIG occupe 3,73 Mio et le GeoJSON préparatoire des bâtiments 0,71 Mio. Ces tailles décrivent la livraison présente; aucun temps d’accès réseau ou de rendu client n’a été mesuré.",
'À COMPLÉTER / VÉRIFIER — remplacer l’état d’avancement':
"Travaux restant à conduire : déployer GeoServer et MapStore; renseigner les URL de services; convertir les bâtiments en 3D Tiles; construire le terrain 3D; puis mesurer la disponibilité, l’interrogation, la fidélité géométrique et les temps de chargement selon un protocole reproductible.",
'À COMPLÉTER / VÉRIFIER — rédiger après obtention des résultats de résolution':
"État des réponses : le corpus est techniquement exploitable pour un prédiagnostic topographique, mais la validation altimétrique indépendante reste trop limitée. Le cas du Vatencul fournit des objets de présélection reproductibles à 0,50 m, sans produire d’aléa ni de risque. L’hypothèse relative à la sensibilité de résolution ne peut pas être réévaluée sur 73,061 ha avec cette livraison. La faisabilité documentaire du paquet WebSIG est établie, contrairement à son fonctionnement déployé.",
'À COMPLÉTER / VÉRIFIER — compléter cette section avec les résultats recalculés à 1 m et 2 m':
"La sensibilité à 1 m et 2 m ne peut pas être discutée sur la nouvelle emprise faute de produits recalculés. Les anciennes valeurs de distance et de longueur obtenues sur une autre emprise sont exclues. Le résultat disponible porte uniquement sur le contraste entre la délimitation hydroconditionnée et le diagnostic D8 brut à 0,50 m.",
'À COMPLÉTER / VÉRIFIER — compléter à partir des essais fonctionnels réels':
"L’intégrité et l’organisation du paquet de publication sont vérifiées, mais aucun essai fonctionnel d’une instance GeoServer/MapStore déployée n’est disponible. Les conclusions doivent donc rester limitées à la préparation des données et des configurations.",
'À COMPLÉTER / VÉRIFIER — remplacer cette discussion issue de l’ancien cas local':
"Sur la nouvelle emprise, les 11,411 km d’axes, 119 bâtiments proches, 66 croisements routiers et 188 dépressions constituent des priorités de contrôle de terrain. Leur portée reste topographique. Aucun de ces dénombrements ne mesure un débit, une hauteur d’eau, une probabilité d’occurrence ou un dommage.",
'À COMPLÉTER / VÉRIFIER — finaliser la conclusion après les recalculs':
"La conclusion est arrêtée au niveau de preuve disponible : diagnostic topographique vérifié à 0,50 m et paquet de diffusion préparé. Les résultats de sensibilité à 1 m et 2 m, la simulation hydraulique et la validation fonctionnelle de MapStore restent hors du champ démontré par la livraison contrôlée."}
for p in doc.paragraphs:
    for pref,new in marker_updates.items():
        if p.text.strip().startswith(pref): p.text=new

# Correct publication inventory and status using exact current text.
replace_prefix('Les produits annoncés comprennent cinq Cloud Optimized GeoTIFF',
"Le paquet de diffusion comprend cinq Cloud Optimized GeoTIFF (MNT, pente, ombrage, aire contributive et profondeur de remplissage), un GeoPackage de 20 couches, un catalogue JSON de sept entrées, une configuration modèle GeoServer/MapStore et un GeoJSON de bâtiments extrudables nécessitant encore une conversion en 3D Tiles. Les sources analytiques restent séparées des dérivés web.")
replace_prefix('Les résultats ont été regroupés dans un paquet de publication',
"Le paquet de publication effectivement préparé contient cinq rasters Cloud Optimized GeoTIFF, un GeoPackage de 20 couches, un catalogue de couches et une configuration MapStore modèle. Les rasters proposés à GeoServer sont le MNT, la pente, l’ombrage, l’aire contributive D8 et la profondeur de remplissage. Les transformations destinées au Web ne remplacent pas les fichiers utilisés dans les calculs.")
replace_prefix('Les couches sont organisées en quatre ensembles',
"Les couches vectorielles couvrent le bassin et l’exutoire, le Vatencul visible et ses raccords, le chemin aval, les axes, les dépressions, le bâti, les routes, les ponts, l’hydrographie, la végétation, le MOS, le PLU et le PPRI. Le catalogue prévoit une diffusion raster en WMS/WMTS et vectorielle en WFS ou OGC API Features. Cette organisation est documentée mais n’équivaut pas à un service déployé.")
# There are two 'La branche 2D...' paragraphs; update both by position within section semantics.
for p in doc.paragraphs:
    if p.text.strip().startswith('La branche 2D est préparée pour une publication par GeoServer'):
        p.text="La branche 2D est préparée pour GeoServer puis MapStore: les vecteurs peuvent être publiés en WMS/WFS ou OGC API Features, et les rasters en WMS/WMTS. Les adresses de services restent des variables à remplacer dans la configuration modèle. Aucune instance déployée n’est fournie dans la livraison."
    if p.text.strip().startswith('La branche 3D est prévu sur le démonstrateur'):
        p.text="La branche 3D fournit un GeoJSON de bâtiments extrudables. Sa conversion en 3D Tiles, l’hébergement de ces tuiles et leur raccordement à MapStore restent nécessaires. Le terrain Cesium et une chaîne complète de visualisation 3D ne sont pas livrés ni testés; aucune performance de chargement ou amélioration de compréhension ne peut donc être affirmée."
    if p.text.strip().startswith('À ce stade, les données analytiques, les dérivés de publication'):
        p.text="La livraison établit la disponibilité et l’intégrité d’un paquet cohérent de préparation à la publication 2D et 3D. Elle ne démontre pas le fonctionnement complet du démonstrateur, car l’instance GeoServer/MapStore, le terrain Cesium et les 3D Tiles ne sont pas fournis comme une chaîne déployée et testée."

replace_prefix('Le corpus général fournit une base techniquement exploitable, sous réserve d’une validation altimétrique externe limitée.',
"Le corpus général fournit une base techniquement exploitable, sous réserve du contrôle altimétrique externe limité. Sur 73,061 ha, le cas du Vatencul produit 11,411 km d’axes D8 bruts, présélectionne 119 bâtiments et 66 tronçons routiers, et identifie 188 dépressions candidates. L’échec du terrain brut à restituer l’aire contributive à l’exutoire souligne le rôle déterminant des continuités souterraines. Les produits WebSIG et HEC-RAS sont préparatoires; la sensibilité à 1 m et 2 m, la simulation hydraulique et le déploiement MapStore ne sont pas démontrés.")

# Front matter grammar.
for p in doc.paragraphs:
    p.text=p.text.replace('la vérification fonctionnelle complète de MapStore doivent encore être finalisées','la vérification fonctionnelle complète de MapStore restent à finaliser')
    p.text=p.text.replace('the complete functional verification of MapStore must still be finalised','the complete functional verification of MapStore remain to be completed')

# Add only figures whose values and labels are internally coherent.
add_figure_after(p_axes,base/'figures/carte_diagnostic_vatencul.png',
'Figure 4.X - Relief, axes D8 bruts, dépressions candidates et continuités documentées dans le bassin du Vatencul.')
add_figure_after(p_land,base/'figures/occupation_sol_mos2025.png',
'Figure 4.X - Dix principales occupations du sol dans le bassin du Vatencul, MOS 2025.',15.0)
add_figure_after(p_rain_result,base/'figures/hyetogramme_gometz_2024_10_09.png',
'Figure 4.X - Hyétogramme historique observé à Gometz-le-Châtel du 9 au 10 octobre 2024, utilisé comme donnée préparatoire.')

# Refresh Word fields on open.
settings=doc.settings._element
update=OxmlElement('w:updateFields'); update.set('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val','true'); settings.append(update)
doc.save(out)

# Structural verification.
check=Document(out)
texts=[p.text.strip() for p in check.paragraphs if p.text.strip()]
print('saved',out,'MiB',round(out.stat().st_size/1024/1024,2))
print('paragraphs',len(check.paragraphs),'tables',len(check.tables),'images',len(check.inline_shapes))
print('unresolved markers',sum('À COMPLÉTER / VÉRIFIER' in t for t in texts))
for t in texts:
    if t.startswith(('3.9.','3.10.','4.4.')): print(t)


In [ ]:
from docx import Document
from pathlib import Path
out=Path('/workspace/Memoire_NAOMIS_restructure_usages_integres_MAJ_Cas_usage_Resultats.docx')
doc=Document(out)
for i,p in enumerate(doc.paragraphs):
    if 'À COMPLÉTER / VÉRIFIER' in p.text:
        print(i,p.text)


In [ ]:
from docx import Document
from pathlib import Path
out=Path('/workspace/Memoire_NAOMIS_restructure_usages_integres_MAJ_Cas_usage_Resultats.docx')
doc=Document(out)
old='À COMPLÉTER / VÉRIFIER — recalculer sur les 73,061 ha les indicateurs à 1 m et 2 m : écarts altimétriques, pente, longueur et recouvrement des axes, distances, bâtiments proches, intersections routières, tailles, durées et mémoire.'
new='Analyse non disponible dans la livraison contrôlée : les indicateurs à 1 m et 2 m devront être recalculés sur les 73,061 ha avant toute conclusion sur la sensibilité de l’altitude, des pentes, des axes, des croisements et des ressources.'
hits=0
for p in doc.paragraphs:
    if p.text.strip()==old:
        p.text=new; hits+=1
doc.save(out)
check=Document(out)
markers=[p.text for p in check.paragraphs if 'À COMPLÉTER / VÉRIFIER' in p.text]
print('replacements',hits,'remaining markers',len(markers),'paragraphs',len(check.paragraphs),'tables',len(check.tables),'images',len(check.inline_shapes))
print('output',out)


In [ ]:
from docx import Document
from pathlib import Path
out=Path('/workspace/Memoire_NAOMIS_restructure_usages_integres_MAJ_Cas_usage_Resultats.docx')
doc=Document(out)
# Final semantic checks for the expected evidence and limitations.
required=['73,061 ha','11,411 km','119, soit 29,9 %','66, soit 36,1 %','188 dépressions','38 empreintes','Aucune simulation défendable','Aucune instance déployée']
text='\n'.join(p.text for p in doc.paragraphs)
print({s:(s in text) for s in required})
print('file_exists',out.exists(),'bytes',out.stat().st_size)
